In [6]:
import numpy as np
from sklearn.model_selection import train_test_split

# 1. D님이 넘겨준 npy 파일 로드
X_raw = np.load('X.npy')
y = np.load('y.npy')

print(f"전체 데이터 형태: X={X_raw.shape}, y={y.shape}")

# 2. Train / Test 분할 (8:2 비율, 클래스 비율 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

전체 데이터 형태: X=(2827876, 70), y=(2827876,)


In [7]:
from sklearn.preprocessing import RobustScaler

# 네트워크 데이터의 극단적인 이상치(Outlier)에 강한 RobustScaler 사용
scaler = RobustScaler()

# X_train에만 fit(기준 설정)과 transform(변환)을 동시에 적용
X_train_scaled = scaler.fit_transform(X_train)

# X_test는 절대 fit하지 않고, Train의 기준에 맞춰 transform(변환)만 적용!
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 1. 평가할 모델들 딕셔너리로 정의
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    # "SVM": SVC(kernel='rbf', probability=True, random_state=42) # 매우 오래 걸릴 수 있으니 유의!
}

# 2. 성능 비교를 담을 리스트
performance_records = []

# 3. 모델별 학습 및 기본 평가 루프
for name, model in models.items():
    print(f"\n[{name}] 학습 시작...")
    model.fit(X_train_scaled, y_train) # 학습
    
    y_pred = model.predict(X_test_scaled) # 테스트 예측
    
    # 평가지표 계산
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # 혼동 행렬 추출 (0: 정상, 1: 공격 이라고 가정)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) # 오탐률(False Positive Rate) 계산
    
    performance_records.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "False Positive Rate (FPR)": fpr
    })
    print(f"{name} 완료! 오탐률(FPR): {fpr*100:.4f}%")





[Logistic Regression] 학습 시작...


c:\python_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression 완료! 오탐률(FPR): 10.1188%

[Random Forest] 학습 시작...
Random Forest 완료! 오탐률(FPR): 0.0658%

[Gradient Boosting] 학습 시작...
Gradient Boosting 완료! 오탐률(FPR): 0.1917%


KeyError: 'FPR'

In [10]:
!pip install tabulate


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd

# 성능 비교 DataFrame 생성
df_performance = pd.DataFrame(performance_records)

# 💡 'FPR' 대신 딕셔너리에 넣었던 정확한 컬럼명인 'False Positive Rate (FPR)'로 수정합니다.
df_performance_sorted = df_performance.sort_values(by=['False Positive Rate (FPR)', 'F1-Score'], ascending=[True, False])

print("\n📊 [모델별 성능 비교표]")
print(df_performance_sorted.to_markdown(index=False))


📊 [모델별 성능 비교표]
| Model               |   Accuracy |   Precision |   Recall |   F1-Score |   False Positive Rate (FPR) |
|:--------------------|-----------:|------------:|---------:|-----------:|----------------------------:|
| Random Forest       |   0.99895  |    0.99895  | 0.99895  |   0.99895  |                 0.000658206 |
| Gradient Boosting   |   0.996011 |    0.996007 | 0.996011 |   0.996008 |                 0.00191738  |
| Logistic Regression |   0.810646 |    0.801321 | 0.810646 |   0.805281 |                 0.101188    |


In [12]:
# Random Forest를 최종 모델로 선정
best_model = models["Random Forest"]

# 기본 예측(predict) 대신 예측 확률(predict_proba)을 가져옴
y_pred_prob = best_model.predict_proba(X_test_scaled)[:, 1] # 클래스 1(공격)일 확률

# 기본 임계값은 0.5 (50%만 넘어도 공격으로 간주)
# 오탐을 줄이기 위해 '매우 확실할 때만(예: 85% 이상)' 공격으로 간주하도록 깐깐하게 변경
CUSTOM_THRESHOLD = 0.85 

# 새로운 임계값 적용
y_pred_strict = (y_pred_prob >= CUSTOM_THRESHOLD).astype(int)

# 깐깐해진 예측 결과로 다시 혼동 행렬 확인
new_cm = confusion_matrix(y_test, y_pred_strict)
tn, fp, fn, tp = new_cm.ravel()
new_fpr = fp / (fp + tn)

print(f"\n[임계값 {CUSTOM_THRESHOLD} 적용 후]")
print(f"오탐(False Positive) 건수: {fp}건")
print(f"새로운 오탐률(FPR): {new_fpr*100:.4f}%")


[임계값 0.85 적용 후]
오탐(False Positive) 건수: 229건
새로운 오탐률(FPR): 0.0504%


In [ ]:
import joblib

# 1. 가장 성능이 좋은 Random Forest를 최종 모델로 선택
# 윤호 님이 백엔드 서버에서 모델을 재학습할 필요 없이 이 파일만 불러와서 실시간 탐지에 사용함
final_best_model = models["Random Forest"]

# 2. 모델과 스케일러를 파일로 저장
joblib.dump(final_best_model, 'ids_rf_model.joblib')
joblib.dump(scaler, 'ids_robust_scaler.joblib')

print("최종 모델 및 스케일러 저장 완료!")

최종 모델 및 스케일러 저장 완료!
